# 07 forge boot

Question: does `golem run` boot the Forge server of `tests/fetch.py` unchanged in its embedded JVM
with the bridge mod connected to the real natives of `golem-bridge`, answer the command `golem` of
the core plugin on the console and to a player, and stop on a signal through the server thread with
the worlds saved, and does a server without the bridge mod still stop and save on a signal.

The notebook builds `golem` with cargo and the bridge with Gradle, then boots copies of the
installed server twice under `golem run`, never under the java launcher. The first boot has the mod
in `mods/`: a player of `tests/harness/minecraft.py` joins, chats and runs `/golem source` and
`/golem status`, the console runs both too, and `SIGTERM` stops the server while the player is
still online, with a `SIGINT` behind it that must change nothing. The second boot has no mod, so
`SIGTERM` must end the JVM through `System.exit` and the shutdown hook of the server, as under
java. `okf/jvm-host.md` describes the stop path and `okf/plugins/core.md` the command.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# Gradle, a path relative to the root or a name on PATH. tests/fetch.py installs it here.
gradle = "tests/.cache/gradle-8.14.3/bin/gradle"
# The JDK 17 whose libjvm golem loads, which also runs Gradle.
java_home = "tests/.cache/jdk-17.0.21+9"
# The Forge server that tests/fetch.py installs. The notebook boots copies of it alone.
forge_server = "tests/.cache/forge-server"
# The port of the copies of the server, which bind 127.0.0.1 alone.
server_port = 25655
# True passes --offline to Gradle, for a machine whose Gradle cache is warm.
gradle_offline = False
# The seconds a boot may take to reach Done, and a stop to end.
boot_timeout = 300

# The pins.
forge_version = "47.4.18"
minecraft_version = "1.20.1"

## Setup

The cell finds the repository and the downloads, builds `golem` with cargo and reads what
`golem source` names: the version, the repository and the commit, which `build.rs` takes from
`git rev-parse HEAD` unless `GOLEM_COMMIT` is set.

In [ ]:
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import threading
import time
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run
from harness.minecraft import Client, offline_uuid

P = paths(golem_root)


def located(value):
    # A value with a slash is a path relative to the root, a bare name is looked up on PATH.
    if "/" in value:
        return (P.root / value).resolve()
    found = shutil.which(value)
    assert found, "%s is not on PATH" % value
    return Path(found)


GRADLE = located(gradle)
JAVA_HOME = located(java_home)
FORGE = located(forge_server)
for needed in (GRADLE, JAVA_HOME / "bin" / "java", FORGE):
    assert needed.exists(), "%s is missing, python3 -I tests/fetch.py installs it" % needed.name
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-07-"))
ENVIRONMENT = dict(os.environ, **QUIET, JAVA_HOME=str(JAVA_HOME))
WORKSPACE = tomllib.loads((P.root / "Cargo.toml").read_text())["workspace"]["package"]


def shown(text):
    # The text with the temporary directory and the root replaced, so no machine path is printed.
    return text.replace(str(SCRATCH), "<scratch>").replace(str(P.root), "<root>")


run(["cargo", "build", "--quiet", "-p", "golem"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
GOLEM = Path(metadata["target_directory"]) / "debug" / "golem"
assert GOLEM.is_file(), "cargo built no golem"
source = dict(line.split(" ", 1) for line in run([GOLEM, "source"], check=True).stdout.splitlines())
COMMIT = source["commit"]
print("python:", sys.version.split()[0])
print("golem source:", source)
assert source["version"] == WORKSPACE["version"] and source["repository"] == WORKSPACE["repository"], source
assert len(COMMIT) == 40 or COMMIT == "unknown", COMMIT

## The bridge

The cell builds a copy of `bridge/` with Gradle, as `05_bridge_build` does and checks in depth,
and keeps `golem-host.jar` and the mod.

In [ ]:
TREE = SCRATCH / "tree"
shutil.copytree(P.root / "bridge", TREE / "bridge", ignore=shutil.ignore_patterns("build", ".gradle"))
shutil.copy2(P.root / "Cargo.toml", TREE / "Cargo.toml")
command = [GRADLE, "-p", TREE / "bridge", "--console=plain", "-Porg.gradle.java.installations.paths=%s" % JAVA_HOME,
           "build"] + (["--offline"] if gradle_offline else [])
started = time.monotonic()
build = subprocess.run([str(part) for part in command], capture_output=True, text=True, env=ENVIRONMENT, timeout=1700)
print("gradle build: exit %d after %.0f s" % (build.returncode, time.monotonic() - started))
assert build.returncode == 0, "the build failed:\n" + build.stdout[-6000:] + build.stderr[-6000:]
HOST_JAR = TREE / "bridge" / "host" / "build" / "libs" / "golem-host.jar"
MOD_JAR = TREE / "bridge" / "mod" / "build" / "libs" / ("golem-bridge-%s.jar" % WORKSPACE["version"])
assert HOST_JAR.is_file() and MOD_JAR.is_file(), (HOST_JAR, MOD_JAR)

## A copy of the server under golem

The cell copies the installed server without its world, logs, crash reports and mods and sets the
port of the copy. A boot runs `golem run` on the copy with the Java home, the host jar and `nogui`
after `--`, reads its output on a thread, waits for `Done` and answers what it printed and logged.
Every line of the host starts with `[golem]` on standard error, which the boot reads together with
the console of the server.

In [ ]:
TOP = {"world", "logs", "crash-reports", "mods"}


def copy_server(name, with_mod):
    server = SCRATCH / name
    shutil.copytree(FORGE, server, symlinks=True,
                    ignore=lambda directory, names: [entry for entry in names if Path(directory) == FORGE
                                                     and (entry in TOP or entry.endswith(".log"))])
    (server / "mods").mkdir()
    if with_mod:
        shutil.copy2(MOD_JAR, server / "mods" / MOD_JAR.name)
    settings = [line for line in (server / "server.properties").read_text().splitlines()
                if not line.startswith(("server-port=", "server-ip=", "online-mode="))]
    settings += ["server-port=%d" % server_port, "server-ip=127.0.0.1", "online-mode=false"]
    (server / "server.properties").write_text("\n".join(settings) + "\n")
    return server


class Boot:
    # One run of golem on a copy of the server, with its output read on a thread as it comes.

    def __init__(self, server):
        self.server = server
        command = [GOLEM, "run", "--server", server, "--java-home", JAVA_HOME, "--host-jar", HOST_JAR, "--", "nogui"]
        self.lines = []
        self.process = subprocess.Popen([str(part) for part in command], cwd=SCRATCH, stdin=subprocess.PIPE,
                                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENVIRONMENT)
        self.reader = threading.Thread(target=self.pump, daemon=True)
        self.reader.start()
        self.started = time.monotonic()
        self.expect("]: Done (", timeout=boot_timeout)
        self.done = time.monotonic() - self.started

    def pump(self):
        for line in self.process.stdout:
            self.lines.append(line.rstrip("\n"))

    def tail(self, count=60):
        return shown("\n".join(self.lines[-count:]))

    def expect(self, text, count=1, timeout=60):
        # Waits until `count` lines of the output hold `text`, and answers the last of them.
        deadline = time.monotonic() + timeout
        while True:
            found = [line for line in list(self.lines) if text in line]
            if len(found) >= count:
                return found[-1]
            if self.process.poll() is not None:
                raise AssertionError("golem ended with %s before %r:\n%s" % (self.process.returncode, text, self.tail()))
            if time.monotonic() > deadline:
                self.process.kill()
                raise AssertionError("no %r after %d s:\n%s" % (text, timeout, self.tail()))
            time.sleep(0.2)

    def console(self, line):
        self.process.stdin.write(line + "\n")
        self.process.stdin.flush()

    def signal(self, number):
        self.process.send_signal(number)

    def wait(self):
        try:
            self.code = self.process.wait(boot_timeout)
        except subprocess.TimeoutExpired:
            self.process.kill()
            raise AssertionError("golem did not end:\n" + self.tail())
        self.reader.join(10)
        self.latest = (self.server / "logs" / "latest.log").read_text(errors="replace")
        crashes = self.server / "crash-reports"
        self.crashes = sorted(path.name for path in crashes.iterdir()) if crashes.is_dir() else []
        self.fatal = [line for line in self.latest.splitlines() if "/FATAL]" in line]
        print("exit %s, Done after %.1f s, FATAL lines %d, crash reports %d" % (
            self.code, self.done, len(self.fatal), len(self.crashes)))
        return self

    def host(self):
        # The lines of the host, without their prefix.
        return [line[len("[golem] "):] for line in self.lines if line.startswith("[golem] ")]


print("boot ready")

## The bridge connects

The server with the mod must reach `Done`, and the core plugin must log once that the bridge is
connected with the versions of Forge and Minecraft. The host names its version, its commit and the
main class of Forge before the JVM runs. On the console, `golem source` must name the license, the
repository and the tree of the commit of the binary, which section 13 of the AGPL asks for, and
`golem status` the ticks that the bridge counted.

In [ ]:
WITH_MOD = copy_server("with-mod", with_mod=True)
on = Boot(WITH_MOD)
on.expect("[golem] the bridge is connected: Forge %s, Minecraft %s" % (forge_version, minecraft_version))
first = on.host()[0]
print(shown(first))
assert first.startswith("Golem %s at %s runs cpw.mods.bootstraplauncher.BootstrapLauncher of " % (
    WORKSPACE["version"], COMMIT)), first
tree = "%s/tree/%s" % (WORKSPACE["repository"], COMMIT) if COMMIT != "unknown" else WORKSPACE["repository"]
on.console("golem source")
on.expect("Golem %s is free software under the GNU Affero General Public License 3.0 only, AGPL-3.0-only." % WORKSPACE["version"])
on.expect("Repository: " + WORKSPACE["repository"])
if COMMIT != "unknown":
    on.expect("Source of this build: " + tree)
on.console("golem status")
status = on.expect("Golem %s: TPS " % WORKSPACE["version"])
print(status.split("]: ", 1)[1])
connected = [line for line in on.host() if line.startswith("the bridge is connected")]
assert len(connected) == 1, connected

## A player

A player joins in offline mode, chats and runs `/golem source` and `/golem status`. The chat must
reach the log of the server, so the natives called on the worker thread of Forge let it pass, and
the player must receive the lines of the core plugin as system messages, with the link to the tree
of the commit.

In [ ]:
player = Client(server_port, "Tester")
player.login()
on.expect("Tester joined the game")
assert player.uuid == offline_uuid("Tester"), player.uuid
player.chat("hello golem")
on.expect("<Tester> hello golem")
player.command("golem source")
assert player.heard("AGPL-3.0-only"), "the player never heard the license: %s" % player.messages
if COMMIT != "unknown":
    assert player.heard(tree), "the player never heard the tree of the commit: %s" % player.messages
player.command("golem status")
assert player.heard("TPS"), "the player never heard the status: %s" % player.messages
print("system messages of the player: %d" % len(player.messages))

## The stop by a signal

A `SIGQUIT` first must print the dump of every thread on the console and stop nothing, as it does
under java, although `-Xrs` takes the signal from the JVM. Then `SIGTERM` must queue
`save-all flush` and `stop` for the server thread, and a `SIGINT` behind it must only log that the
stop is queued. The mod runs both commands as the console at the end of the
next tick, so the game is saved before the server stops, and the player still online is
disconnected. `golem run` must exit with 0 after `DestroyJavaVM` answered 0, with no fatal line and
no crash report.

In [ ]:
on.signal(signal.SIGQUIT)
on.expect("[golem] SIGQUIT: printed the dump of every thread", timeout=30)
dump = [line for line in on.lines if line.startswith("Full thread dump OpenJDK")]
print(dump[0] if dump else "no dump")
assert dump, "SIGQUIT printed no dump of the threads"
assert any(line.startswith('"Server thread"') for line in on.lines), "the dump lacks the server thread"
time.sleep(2)
assert on.process.poll() is None, "SIGQUIT ended golem"
on.signal(signal.SIGTERM)
on.expect("[golem] SIGTERM: queued save-all flush and stop on the server thread", timeout=10)
on.signal(signal.SIGINT)
on.expect("[golem] SIGINT: the stop is already queued, waiting for the server to end", timeout=10)
on.wait()
assert player.closed.wait(30), "the player was never disconnected"
player.close()
print(shown("\n".join(on.host()[-4:])))
assert on.code == 0, "golem exited %s:\n%s" % (on.code, on.tail())
assert on.host()[-1] == "the server ended: main returned and DestroyJavaVM answered 0", on.host()
assert not on.fatal and not on.crashes, (on.fatal, on.crashes)
order = [text for text in ("Saving the game", "Stopping the server", "Saving worlds", "All dimensions are saved")
         if text in on.latest]
assert order == ["Saving the game", "Stopping the server", "Saving worlds", "All dimensions are saved"], order
assert on.latest.index("Saving the game") < on.latest.index("Stopping the server"), "save-all ran after stop"
assert (WITH_MOD / "world" / "level.dat").is_file(), "the world was not saved"

## A server without the bridge

The second copy has no mod in `mods/`, so the natives are registered but nothing calls them and no
tick drains the queue. The server must still reach `Done` as a stock Forge server, and `SIGTERM`
must end the JVM through `System.exit(143)`: the shutdown hook of the server stops it and saves the
worlds, and the exit hook of the host ends golem with 143, as the java launcher would.

In [ ]:
WITHOUT_MOD = copy_server("without-mod", with_mod=False)
off = Boot(WITHOUT_MOD)
time.sleep(2)
off.signal(signal.SIGTERM)
off.expect("[golem] SIGTERM: the bridge is not connected, so the JVM exits through the shutdown hook of the server",
           timeout=10)
off.wait()
print(shown("\n".join(off.host())))
assert not any(line.startswith("the bridge is connected") for line in off.host()), off.host()
assert off.code == 128 + signal.SIGTERM, "golem exited %s:\n%s" % (off.code, off.tail())
assert off.host()[-1] == "the JVM exits with status 143", off.host()
assert not off.fatal and not off.crashes, (off.fatal, off.crashes)
for text in ("Stopping server", "Saving worlds", "All dimensions are saved"):
    assert text in off.latest, "the log lacks %r" % text
assert (WITHOUT_MOD / "world" / "level.dat").is_file(), "the world was not saved"

## Verdict

The cell clears the temporary directory. A run that reaches it passed.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: golem run boots Forge %s with the bridge connected in %.0f s, answers golem source and status on the "
      "console and to a player, dumps its threads on SIGQUIT, stops on SIGTERM through the server thread with exit 0, and stops a server without "
      "the bridge through its shutdown hook with exit 143" % (forge_version, on.done))